# Module 3: State Reducers & Merge Mechanics

In this notebook, we will explore:
1. **Overwrite Default**: Verifying why standard lists get overwritten by default.
2. **Custom Reducer**: Creating a list-appender reducer using python type annotations.
3. **add_messages**: Setting up conversational graphs using built-in message reducers.
4. **Upserting (ID Replacement)**: Updating existing messages inside the state using unique message IDs.

### Step 1: Initialize Chat Model Connection

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(dotenv_path="../../../langchain/.env")

model = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0.3,
)
print("Model client connected!")

Model client connected!


---
## 1. Proving list overwriting by default

Let's see what happens if we use a standard `TypedDict` containing a list of logs.

In [6]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class OverwriteState(TypedDict):
    logs: list[str]

def node_one(state: OverwriteState) -> dict:
    return {"logs": ["Started Task 1"]}

def node_two(state: OverwriteState) -> dict:
    return {"logs": ["Finished Task 2"]}

# Build graph
builder_test = StateGraph(OverwriteState)
builder_test.add_node("n1", node_one)
builder_test.add_node("n2", node_two)
builder_test.add_edge(START, "n1")
builder_test.add_edge("n1", "n2")
builder_test.add_edge("n2", END)

graph_test = builder_test.compile()
res = graph_test.invoke({"logs": ["Init Log"]})
print("Final State 'logs' value:", res["logs"])

Final State 'logs' value: ['Finished Task 2']


As we can see, the final log list only contains the updates from `node_two`! The previous logs were completely overwritten.

---
## 2. Implementing a Custom Reducer

Now, we will define a reducer function that performs list addition (`current + update`), and attach it to the `logs` key using `Annotated`.

In [7]:
from typing import Annotated

# 1. Define the reducer logic
def list_accumulator_reducer(current: list[str], update: list[str]) -> list[str]:
    # Append new items to the existing list
    return current + update

# 2. Declare the State using Annotated
class AccumulatorState(TypedDict):
    logs: Annotated[list[str], list_accumulator_reducer]

# We redefine the graph nodes using this new state
def accumulator_node_one(state: AccumulatorState) -> dict:
    return {"logs": ["Started Task 1"]}

def accumulator_node_two(state: AccumulatorState) -> dict:
    return {"logs": ["Finished Task 2"]}

builder_acc = StateGraph(AccumulatorState)
builder_acc.add_node("n1", accumulator_node_one)
builder_acc.add_node("n2", accumulator_node_two)
builder_acc.add_edge(START, "n1")
builder_acc.add_edge("n1", "n2")
builder_acc.add_edge("n2", END)

graph_acc = builder_acc.compile()
res_acc = graph_acc.invoke({"logs": ["Init Log"]})
print("Final Accumulated Logs:", res_acc["logs"])

Final Accumulated Logs: ['Init Log', 'Started Task 1', 'Finished Task 2']


Perfect! The logs are now preserved and accumulated correctly.

---
## 3. Conversational Memory: Using `MessagesState`

Instead of writing custom list appenders for chat histories, we inherit from `MessagesState` which handles message list merging out-of-the-box.

In [10]:
from langgraph.graph import MessagesState
from langchain_core.messages import SystemMessage, HumanMessage

# CustomState inherits the 'messages' list field from MessagesState
class ChatGraphState(MessagesState):
    user_profile: str  # Extra key we can define

def responder_node(state: ChatGraphState) -> dict:
    print("--- Responder Node Executing ---")
    # Access the accumulated messages list
    history = state["messages"]
    profile = state.get("user_profile", "Guest")
    
    # Append a greeting system context
    context = [
        SystemMessage(content=f"You are a conversational guide assisting {profile}.")
    ] + history
    
    response = model.invoke(context)
    
    # Return message update to be merged by the 'add_messages' reducer
    return {"messages": [response]}

builder_chat = StateGraph(ChatGraphState)
builder_chat.add_node("assistant", responder_node)
builder_chat.add_edge(START, "assistant")
builder_chat.add_edge("assistant", END)

chat_graph = builder_chat.compile()
print("Conversational MessagesState graph compiled!")

Conversational MessagesState graph compiled!


In [11]:
# Running a call
input_payload = {
    "user_profile": "Developer Alice",
    "messages": [HumanMessage(content="Hi! Recommend one good backend python framework.")]
}

final_state = chat_graph.invoke(input_payload)
print("\n=== FINAL CONVERSATIONAL HISTORY ===")
for msg in final_state["messages"]:
    print(f"[{type(msg).__name__}]: {msg.content}")

--- Responder Node Executing ---

=== FINAL CONVERSATIONAL HISTORY ===
[HumanMessage]: Hi! Recommend one good backend python framework.
[AIMessage]: Hey there!👋  
If I had to pick **one** backend Python framework that balances simplicity, flexibility, and a strong ecosystem, I’d go with **FastAPI**.

### Why FastAPI?

| Feature | What it means for you | Why it matters |
|---------|----------------------|----------------|
| **Performance** | Built on **Starlette** (ASGI) and **Pydantic**, it’s one of the fastest Python web frameworks (often beating Node.js & Go in benchmarks). | Handles high traffic with fewer workers—great for scaling APIs. |
| **Async‑first** | First‑class support for `async def` endpoints, async DB drivers, and async I/O. | You can write non‑blocking code without extra libraries. |
| **Automatic Docs** | Generates **OpenAPI** and **ReDoc** docs automatically from your type hints. | No extra work to expose interactive Swagger UI or Redoc for your API. |
| **Data Valid

---
## 4. Message Upserting (ID Replacement)

Because the built-in `add_messages` reducer checks message IDs, we can update or overwrite specific entries in the history list by passing a message containing a matching ID.

In [12]:
from langchain_core.messages import AIMessage

# Create an initial message list state
initial_messages = [
    HumanMessage(content="Hello assistant!", id="msg_user_1"),
    AIMessage(content="Hello! Processing your request...", id="msg_ai_1")
]

print("=== BEFORE STATE MERGE ===")
for msg in initial_messages:
    print(f"ID: {msg.id} | Content: {msg.content}")

# We import the add_messages function directly to demonstrate its standalone behavior:
from langgraph.graph.message import add_messages

# Create an update containing a message with a matching ID ('msg_ai_1')
update_payload = [
    AIMessage(content="Request Processed! The database is online.", id="msg_ai_1")
]

# Run the reducer manually
merged_messages = add_messages(initial_messages, update_payload)

print("\n=== AFTER STATE MERGE ===")
for msg in merged_messages:
    print(f"ID: {msg.id} | Content: {msg.content}")

=== BEFORE STATE MERGE ===
ID: msg_user_1 | Content: Hello assistant!
ID: msg_ai_1 | Content: Hello! Processing your request...

=== AFTER STATE MERGE ===
ID: msg_user_1 | Content: Hello assistant!
ID: msg_ai_1 | Content: Request Processed! The database is online.


Notice that the final list size remains 2, and the content of `msg_ai_1` was updated! This is how you can correct previous agent outputs, overwrite intermediate tool logs, or implement sliding context limits.